# Store Sales Exploratory Data Analysis

This notebook turns the labeled training period into requirements for the
16-day store-and-family forecast. Validation and internal-test dates are fixed
before analysis, keeping their targets out of charts and feature decisions.

**Input:** `data/processed/00_STORE_SALES_EDA.csv`

**Output:** a concise evidence table for the modeling stage.


In [ ]:
from pathlib import Path
import gc

import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import pandas as pd
from IPython.display import display

from store_sales_preprocessing import (
    BASE_KEY,
    CALENDAR_COLUMNS,
    HOLIDAY_FEATURE_COLUMNS,
    OIL_AGE_COLUMNS,
    OIL_FEATURE_COLUMNS,
    OIL_LAG_COLUMNS,
    STORE_OUTPUT_COLUMNS,
    TRAIN_COLUMNS,
    TRANSACTION_FEATURE_COLUMNS,
    TRANSACTION_LAG_COLUMNS,
)

pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

FIGURE_DPI = 150
FIGSIZE_16_9 = (12.8, 7.2)
FIGSIZE_4_3 = (10.0, 7.5)


def format_millions(value: float, _position: int) -> str:
    """Format large chart values as readable millions, such as 25M."""
    return f"{value / 1_000_000:,.0f}M"


def format_millions_precise(value: float, _position: int) -> str:
    """Keep one decimal where sub-million differences matter."""
    return f"{value / 1_000_000:,.1f}M"


DATA_PATH = Path("data/processed/00_STORE_SALES_EDA.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "data/processed/00_STORE_SALES_EDA.csv is not available. Run "
        "01_STORE_SALES_PREPROCESSING.ipynb first."
    )

print(f"Data source: {DATA_PATH}")


## 1. Freeze the evaluation windows

Fix three consecutive periods before looking at target patterns: training for
analysis and fitting, validation for model selection, and one 16-day internal
test. `split_plan` records the exact boundaries.


In [ ]:
TRAIN_END = pd.Timestamp("2017-07-14")
VALIDATION_START = pd.Timestamp("2017-07-15")
VALIDATION_END = pd.Timestamp("2017-07-30")
INTERNAL_TEST_START = pd.Timestamp("2017-07-31")
INTERNAL_TEST_END = pd.Timestamp("2017-08-15")

split_plan = pd.DataFrame(
    [
        {
            "split": "train",
            "start": "start of labeled history",
            "end": TRAIN_END.date().isoformat(),
            "purpose": "EDA, feature development, and model fitting",
        },
        {
            "split": "validation",
            "start": VALIDATION_START.date().isoformat(),
            "end": VALIDATION_END.date().isoformat(),
            "purpose": "model comparison and design decisions",
        },
        {
            "split": "internal_test",
            "start": INTERNAL_TEST_START.date().isoformat(),
            "end": INTERNAL_TEST_END.date().isoformat(),
            "purpose": "one final internal evaluation",
        },
    ]
)

display(split_plan)

## 2. Load the accepted processed dataset

Check the saved header against the 34-column preprocessing contract, then load
the table with compact dtypes. The load summary shows shape, date coverage, and
memory use.


In [ ]:
EXPECTED_COLUMNS = (
    TRAIN_COLUMNS
    + STORE_OUTPUT_COLUMNS
    + CALENDAR_COLUMNS
    + OIL_FEATURE_COLUMNS
    + TRANSACTION_FEATURE_COLUMNS
    + HOLIDAY_FEATURE_COLUMNS
)

saved_columns = pd.read_csv(DATA_PATH, nrows=0).columns.tolist()
if saved_columns != EXPECTED_COLUMNS:
    raise ValueError(
        "The processed CSV does not match the final 34-column contract. "
        "Run 01_STORE_SALES_PREPROCESSING.ipynb from top to bottom before EDA. "
        f"Expected {EXPECTED_COLUMNS}, received {saved_columns}."
    )

CATEGORY_COLUMNS = ["family", "city", "state", "store_type"]
FLOAT32_COLUMNS = [
    *OIL_LAG_COLUMNS,
    *OIL_AGE_COLUMNS,
    *TRANSACTION_LAG_COLUMNS,
]
INTEGER_COLUMNS = [
    "store_nbr",
    "store_cluster",
    *CALENDAR_COLUMNS,
    "transactions_lag_available_count",
    *HOLIDAY_FEATURE_COLUMNS,
]

DTYPES = {
    "id": "int32",
    "sales": "float64",
    "onpromotion": "int32",
    **{column: "float32" for column in FLOAT32_COLUMNS},
    **{column: "category" for column in CATEGORY_COLUMNS},
    **{column: "int16" for column in INTEGER_COLUMNS},
}

data = pd.read_csv(DATA_PATH, dtype=DTYPES, parse_dates=["date"])

memory_mb = data.memory_usage(deep=True).sum() / 1024**2
print(f"Dataset shape : {data.shape[0]:,} rows x {data.shape[1]} columns")
print(f"Date range    : {data['date'].min().date()} to {data['date'].max().date()}")
print(f"Memory usage  : {memory_mb:,.1f} MB")


## 3. Validate structural integrity

`structural_checks` covers business keys, target validity, and entity coverage.
`missing_feature_audit` separates expected historical gaps from data issues,
and `absent_dates` lists unobserved calendar dates.


In [ ]:
nullable_history_columns = [
    *OIL_AGE_COLUMNS,
    *OIL_LAG_COLUMNS,
    *TRANSACTION_LAG_COLUMNS,
]
complete_calendar = pd.date_range(data["date"].min(), data["date"].max(), freq="D")
missing_calendar_dates = complete_calendar.difference(data["date"].unique())

missing_by_column = data.isna().sum()
unexpected_missing = missing_by_column.loc[
    (missing_by_column > 0)
    & (~missing_by_column.index.isin(nullable_history_columns))
]
missing_feature_audit = (
    missing_by_column.loc[missing_by_column > 0]
    .rename("missing_cells")
    .to_frame()
    .assign(missing_rate_pct=lambda frame: 100 * frame["missing_cells"] / len(data))
)

structural_checks = pd.Series(
    {
        "expected_history_gap_cells": int(missing_by_column.sum()),
        "unexpected_missing_cells": int(unexpected_missing.sum()),
        "duplicate_date_store_family": int(data.duplicated(BASE_KEY).sum()),
        "negative_sales_rows": int((data["sales"] < 0).sum()),
        "unique_dates": int(data["date"].nunique()),
        "missing_calendar_dates": int(len(missing_calendar_dates)),
        "unique_stores": int(data["store_nbr"].nunique()),
        "unique_families": int(data["family"].nunique()),
    },
    name="value",
).to_frame()

absent_dates = pd.DataFrame(
    {"absent_date": [date.date().isoformat() for date in missing_calendar_dates]}
)

display(structural_checks)
display(missing_feature_audit)
display(absent_dates)

assert structural_checks.loc["unexpected_missing_cells", "value"] == 0
assert structural_checks.loc["duplicate_date_store_family", "value"] == 0
assert structural_checks.loc["negative_sales_rows", "value"] == 0

print("PASS: structural integrity is ready for training-only EDA.")


## 4. Apply the chronological split

Assign every labeled row to `train`, `validation`, or `internal_test` from the
fixed boundaries. `split_counts` shows complete assignment and 16 dates in each
later window.


In [ ]:
data["data_split"] = "unassigned"
data.loc[data["date"].le(TRAIN_END), "data_split"] = "train"
data.loc[
    data["date"].between(VALIDATION_START, VALIDATION_END),
    "data_split",
] = "validation"
data.loc[data["date"].between(INTERNAL_TEST_START, INTERNAL_TEST_END), "data_split"] = "internal_test"

data["data_split"] = pd.Categorical(
    data["data_split"],
    categories=["train", "validation", "internal_test", "unassigned"],
    ordered=True,
)

split_counts = (
    data.groupby("data_split", observed=False)
    .agg(rows=("id", "size"), observed_dates=("date", "nunique"))
    .reset_index()
)

display(split_counts)

assert int((data["data_split"] == "unassigned").sum()) == 0
assert int(
    split_counts.loc[split_counts["data_split"] == "validation", "observed_dates"].iloc[0]
) == 16
assert int(
    split_counts.loc[split_counts["data_split"] == "internal_test", "observed_dates"].iloc[0]
) == 16

print("PASS: every row belongs to exactly one modeling split.")

## 5. Isolate the training period

From this point, target analysis uses training rows only. `train_summary` shows
date and entity coverage, zero-sales rate, and promotion coverage.


In [ ]:
EDA_COLUMNS = [
    "date",
    "store_nbr",
    "family",
    "sales",
    "onpromotion",
    "day_of_week",
    "is_holiday",
    "is_planned_event",
    "is_national_schedule",
]

train_data = data.loc[data["data_split"] == "train", EDA_COLUMNS].copy()
train_data["is_weekend_group"] = train_data["day_of_week"].isin([6, 7])
del data
gc.collect()

train_summary = pd.Series(
    {
        "rows": len(train_data),
        "start_date": train_data["date"].min().date().isoformat(),
        "end_date": train_data["date"].max().date().isoformat(),
        "observed_dates": train_data["date"].nunique(),
        "stores": train_data["store_nbr"].nunique(),
        "product_families": train_data["family"].nunique(),
        "zero_sales_rate_pct": 100 * train_data["sales"].eq(0).mean(),
        "promotion_rows_pct": 100 * train_data["onpromotion"].gt(0).mean(),
    },
    name="training_value",
).to_frame()

display(train_summary)

## 6. Measure the daily sales pattern

Aggregate training sales by date and add a 28-day moving average to expose the
broader pattern. The chart informs calendar and lag features; the moving average
is used for analysis rather than as a model input.


In [ ]:
daily_sales = train_data.groupby("date", observed=True)["sales"].sum().sort_index()
daily_rolling_28 = daily_sales.rolling("28D", min_periods=7).mean()

fig, ax = plt.subplots(figsize=FIGSIZE_16_9, dpi=FIGURE_DPI)
ax.plot(
    daily_sales.index,
    daily_sales.values,
    color="#94a3b8",
    linewidth=0.8,
    label="Daily total",
)
ax.plot(
    daily_rolling_28.index,
    daily_rolling_28.values,
    color="#0f766e",
    linewidth=2.0,
    label="28-day moving average",
)
ax.set_title("Daily Sales During the Training Period")
ax.set_xlabel("Date")
ax.set_ylabel("Total sales (millions)")
ax.yaxis.set_major_formatter(FuncFormatter(format_millions_precise))
ax.grid(alpha=0.2)
ax.legend()
plt.tight_layout()
plt.show()

## 7. Compare product-family behavior

Compare total sales, row-level averages, and zero-sales rates by product family.
The top-ten chart makes the scale differences visible.


In [ ]:
family_summary = (
    train_data.groupby("family", observed=True)
    .agg(
        total_sales=("sales", "sum"),
        average_daily_store_sales=("sales", "mean"),
        zero_sales_rate_pct=("sales", lambda values: 100 * values.eq(0).mean()),
    )
    .sort_values("total_sales", ascending=False)
)

display(family_summary.head(10))

top_families = family_summary.head(10).sort_values("total_sales")
fig, ax = plt.subplots(figsize=FIGSIZE_4_3, dpi=FIGURE_DPI)
ax.barh(
    top_families.index.astype(str),
    top_families["total_sales"],
    color="#2563eb",
)
ax.set_title("Top 10 Product Families by Total Sales")
ax.set_xlabel("Total sales during the training period (millions)")
ax.xaxis.set_major_formatter(FuncFormatter(format_millions))
ax.grid(axis="x", alpha=0.2)
plt.tight_layout()
plt.show()

## 8. Compare store behavior

`store_summary` compares total and average sales across stores. The top-ten
chart highlights the largest sales populations.


In [ ]:
store_summary = (
    train_data.groupby("store_nbr", observed=True)["sales"]
    .agg(total_sales="sum", average_daily_family_sales="mean")
    .sort_values("total_sales", ascending=False)
)

display(store_summary.head(10))

top_stores = store_summary.head(10).sort_values("total_sales")
fig, ax = plt.subplots(figsize=FIGSIZE_4_3, dpi=FIGURE_DPI)
ax.barh(
    top_stores.index.astype(str),
    top_stores["total_sales"],
    color="#7c3aed",
)
ax.set_title("Top 10 Stores by Total Sales")
ax.set_xlabel("Total sales during the training period (millions)")
ax.xaxis.set_major_formatter(FuncFormatter(format_millions))
ax.set_ylabel("Store number")
ax.grid(axis="x", alpha=0.2)
plt.tight_layout()
plt.show()

## 9. Compare promotion and calendar groups

`promotion_summary` and `calendar_summary` compare counts, means, medians,
variability, and zero-sales rates. Weekend status is derived for this view and
kept out of the model feature set.


In [ ]:
train_data["has_promotion"] = train_data["onpromotion"].gt(0)

promotion_summary = (
    train_data.groupby("has_promotion", observed=True)["sales"]
    .agg(
        rows="size",
        mean_sales="mean",
        median_sales="median",
        zero_sales_rate_pct=lambda values: 100 * values.eq(0).mean(),
    )
    .rename(index={False: "without promotion", True: "with promotion"})
)

calendar_summary = pd.DataFrame(
    {
        "weekday_non_holiday": train_data.loc[
            (~train_data["is_weekend_group"])
            & (train_data["is_holiday"] == 0),
            "sales",
        ].describe(),
        "weekend_non_holiday": train_data.loc[
            train_data["is_weekend_group"]
            & (train_data["is_holiday"] == 0),
            "sales",
        ].describe(),
        "holiday": train_data.loc[
            train_data["is_holiday"] == 1, "sales"
        ].describe(),
    }
).loc[["count", "mean", "50%", "std"]]

print("Promotion summary")
display(promotion_summary)
print("Calendar summary")
display(calendar_summary)

## 10. Define the modeling requirements

`findings` links each training observation to a modeling decision covering
metrics, entity identity, promotions, calendar context, and sales history.


In [ ]:
top_family = str(family_summary.index[0])
top_store = int(store_summary.index[0])
zero_rate = 100 * train_data["sales"].eq(0).mean()
promotion_mean_ratio = (
    promotion_summary.loc["with promotion", "mean_sales"]
    / promotion_summary.loc["without promotion", "mean_sales"]
)

findings = pd.DataFrame(
    {
        "evidence": [
            f"{zero_rate:.2f}% of training rows have zero sales.",
            f"{top_family} has the highest total sales among product families.",
            f"Store {top_store} has the highest total sales among stores.",
            f"Mean sales on promoted rows are {promotion_mean_ratio:.2f}x mean sales on non-promoted rows.",
            "The data contain time, product-family, store, promotion, and calendar patterns.",
        ],
        "modeling_implication": [
            "Use metrics that remain meaningful with zero targets, including RMSLE and WAPE.",
            "The model must distinguish product families.",
            "The model must distinguish stores and their location context.",
            "Promotion count is a reasonable feature, but its association must not be described as causal impact.",
            "The next stage should create exact past-only sales lags under the 16-day forecast contract.",
        ],
    }
)

display(findings)

print("EDA COMPLETE: validation and test sales were not used in target analysis.")

## EDA handoff

The `findings` table links each training-period observation to a modeling
decision. Notebook 03 uses that handoff for feature engineering, chronological
model comparison, and reusable inference.
